In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from tfmap import Atlus
import numpy as np
import polars as pl

In [ ]:
def atlus_to_df(map_obj: Atlus, cell_type: str) -> pl.DataFrame:
    wn = np.linspace(650, 4000, 3475)
    pixel_idx, pixel_pos = list(zip(*map_obj.pixels.items()))
    pixel_x, pixel_y = list(zip(*pixel_pos))
    pixel_df = pl.DataFrame(dict(idx=pixel_idx, pixel_x=pixel_x, pixel_y=pixel_y))

    spectra_idx, spectra = list(zip(*map_obj.spectra_dict.items()))
    spectra_df = pl.DataFrame(np.array(spectra))
    spectra_df.columns = [f"wavenumber_{x:.2f}" for x in wn]
    spectra_df = spectra_df.with_columns(pl.Series(name="idx", values=spectra_idx))

    return (
        pixel_df.join(spectra_df, on="idx")
        .drop("idx")
        .with_columns(cell_type=pl.lit(cell_type))
    )

In [ ]:
ctrl_atlus = Atlus.from_map_filepath(
    "../extra/250511-synchrotron/CTRL sample 1_2H_thp1-ctrl_Tr_32x_2.5stepsize_8resolution_2scans_Ifg.map"
)
ctrl_df = atlus_to_df(ctrl_atlus, "ctrl")
ctrl_atlus

In [ ]:
ctrl_df

In [ ]:
import matplotlib.pyplot as plt
from pybaselines import Baseline


def baseline_correction(atlus, plot_res = False):
    wavenumbers = np.linspace(650, 4000, 3475)
    clip_mask = (900 <= wavenumbers) & (wavenumbers <= 3600)
    wavenumbers = wavenumbers[clip_mask]

    res = None
    baseline_fitter = Baseline(x_data=wavenumbers)
    for example_spectra in atlus.spectra_dict.values():
        example_spectra = 2 - np.log10(example_spectra)
        example_spectra = example_spectra[clip_mask]
        res = baseline_fitter.rubberband(example_spectra, segments=[900, 1900, 2250])[0]
        if plot_res:
            plt.plot(example_spectra - res)
    return res

baseline_correction(ctrl_atlus, True)

In [ ]:
import polars.selectors as cs


def baseline_correction_df(df, plot_res=False):
    clipped_spectra_cols = [
        col
        for col in df.select(cs.contains("wavenumber")).columns
        if 900 <= float(col.split("_")[-1]) <= 3600
    ]
    wavenumbers = [float(col.split("_")[-1]) for col in clipped_spectra_cols]

    baseline_fitter = Baseline(x_data=wavenumbers)
    acc = []
    for example_spectra in df.select(clipped_spectra_cols).to_numpy():
        example_spectra = 2 - np.log10(example_spectra)
        res = baseline_fitter.rubberband(example_spectra, segments=[900, 1900, 2250])[0]
        if plot_res:
            plt.plot(example_spectra - res)
        acc.append(example_spectra - res)
    return pl.concat(
        [
            pl.DataFrame(np.array(acc), schema=clipped_spectra_cols),
            df.select(~cs.contains("wavenumber")),
        ],
        how="horizontal",
    )


baseline_correction_df(ctrl_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import polars.selectors as cs
from sklearn.cluster import KMeans
from typing import Optional


def umap_to_kmean(
    df: pl.DataFrame,
    n_clusters=2,
    n_neighbors=15,
    masks: Optional[list[tuple[float, float]]] = None,
):
    if masks is None:
        masks = [(0.0, 5000.0)]

    cols = [
        col
        for col in df.select(cs.contains("wavenumber")).columns
        if any(low <= float(col.split("_")[-1]) <= high for low, high in masks)
    ]
    X = df.select(cols).to_numpy()
    components = umap.UMAP(densmap=True, n_neighbors=n_neighbors).fit_transform(X)
    split = KMeans(n_clusters=2).fit_predict(components)
    return df.with_columns(pc1=components[:, 0], pc2=components[:, 1], split=split)


def add_umap_components(
    df: pl.DataFrame,
    masks: Optional[list[tuple[float, float]]] = None,
    run_umap=False,
) -> pl.DataFrame:
    if masks is None:
        masks = [(0.0, 5000.0)]

    cols = [
        col
        for col in df.select(cs.contains("wavenumber")).columns
        if any(low <= float(col.split("_")[-1]) <= high for low, high in masks)
    ]
    X = df.select(cols).to_numpy()
    if run_umap:
        components = umap.UMAP(densmap=True, random_state=717).fit_transform(X)
        return df.with_columns(pc1=components[:, 0], pc2=components[:, 1])
    else:
        components = KMeans(n_clusters=2).fit_predict(X)
        return df.with_columns(split=components)


# umap_ctrl_df = add_umap_components(ctrl_df).with_columns(
#     pl.when(pl.col("pc1") < 2.5).then(0).otherwise(1).alias("split")
# )
umap_ctrl_df = add_umap_components(ctrl_df)
ukmap_ctrl_df = umap_to_kmean(ctrl_df)
fig, ax = plt.subplots(1, 2, figsize=(12, 8))
ctrl_atlus.plot_rgb_image(ax=ax[0])
sns.scatterplot(
    data=umap_ctrl_df,
    x="pixel_x",
    y="pixel_y",
    hue="split",
    marker="s",
    linewidth=0,
    s=5,
    alpha=0.7,
    ax=ax[0],
)

ctrl_atlus.plot_rgb_image(ax=ax[1])
sns.scatterplot(
    data=ukmap_ctrl_df,
    x="pixel_x",
    y="pixel_y",
    hue="split",
    marker="s",
    linewidth=0,
    s=5,
    alpha=0.7,
    ax=ax[1],
)
# sns.scatterplot(
#     data=umap_ctrl_df,
#     x="pc1",
#     y="pc2",
#     hue="split",
#     ax=ax[1],
#     # marker="s",
#     # linewidth=0,
#     # s=5,
#     # alpha=0.5
# )

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ctrl_atlus.plot_rgb_image(ax=ax)
sns.scatterplot(
    data=ukmap_ctrl_df.with_columns(
        pl.col("pixel_x").add(5), pl.col("pixel_y").sub(15)
    ),
    x="pixel_x",
    y="pixel_y",
    hue="split",
    marker="s",
    linewidth=0,
    s=5,
    alpha=0.7,
    ax=ax,
)
fig.suptitle("Manually aligned, CTRL cells")

In [ ]:
xs = []
ys = []
for idx, (x, y) in ctrl_atlus.pixels.items():
    xs.append(x)
    ys.append(y)
print([a.item() for a in ctrl_atlus._max_extent()])
print([min(xs).item(), max(xs).item(), min(ys).item(), max(ys).item()])
print(ctrl_atlus.pixels_projected_extent(362, 496))

In [ ]:
lta_atlus = Atlus.from_map_filepath(
    "../extra/250511-synchrotron/LTA sample 2_2h_thp1-lta_Tr_32x_2.5stepsize_8resolution_2scans_Ifg.map"
)
lta_df = atlus_to_df(lta_atlus, "lta")
lta_atlus

In [ ]:
umap_lta_df = add_umap_components(lta_df).with_columns(
    pl.when((1.5 * pl.col("pc1") - pl.col("pc2")) >= 0)
    .then(1)
    .otherwise(0)
    .alias("split")
)
fig, ax = plt.subplots(1, 2, figsize=(12, 8))
lta_atlus.plot_rgb_image(ax=ax[0])
sns.scatterplot(
    data=umap_lta_df,
    x="pixel_x",
    y="pixel_y",
    hue="split",
    marker="s",
    linewidth=0,
    s=5,
    alpha=0.5,
    ax=ax[0],
)
sns.scatterplot(
    data=umap_lta_df,
    x="pc1",
    y="pc2",
    hue="split",
    ax=ax[1],
    # marker="s",
    # linewidth=0,
    # s=5,
    # alpha=0.5
)

In [ ]:
lps_atlus = Atlus.from_map_filepath(
    "../extra/250511-synchrotron/LPS_sample_2_2hr_thp1_lps_Tr_32x_2.5stepsize_8resolution_2scans.map",
    parse_spectra=True,
)
lps_df = atlus_to_df(lps_atlus, "lps")
ukmap_lps_df = umap_to_kmean(lps_df)
lps_atlus

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
lps_atlus.plot_rgb_image(ax=ax)
sns.scatterplot(
    data=ukmap_lps_df.with_columns(
        pl.col("pixel_x").add(5), pl.col("pixel_y").sub(15)
    ),
    x="pixel_x",
    y="pixel_y",
    hue="split",
    marker="s",
    linewidth=0,
    s=5,
    alpha=0.7,
    ax=ax,
)
fig.suptitle("Manually aligned, LPS cells")

In [ ]:
ukmap_lps_df.filter(pl.col("split") == 1).write_csv("../extra/lps_ukmap_df_filtered.csv")

In [ ]:
all_df = pl.concat([ctrl_df, lps_df, lta_df])
all_df.shape

In [ ]:
import umap
import polars.selectors as cs
import seaborn as sns

# components = umap.UMAP(densmap=True).fit_transform(
#     all_df.select(cs.contains("wavenumber")).to_numpy()
# )
fig, ax = plt.subplots(1, 2)
all_umap_df = umap_to_kmean(baseline_correction_df(all_df))
sns.scatterplot(
    data=all_umap_df, x="pc1", y="pc2", hue="cell_type", s=2, alpha=0.8, ax=ax[0]
)
sns.scatterplot(
    data=all_umap_df, x="pc1", y="pc2", hue="split", s=2, alpha=0.8, ax=ax[1]
)

In [ ]:

def align_imgs_post(atlus, umap_df):
    # blur_kernels = [(x, x) for x in range(7, 15, 2)]
    blur_kernels = [(13, 13)]
    masks = [
        [(0.0, 5000.0)],
        [(800, 1800)],
        [(800, 1800), (2800, 3000)],
        [(2800, 3000)],
    ]
    n_cluster_params = [2, 3, 4]
    for mask, blur_kernel_1, blur_kernel_2, n_clusters in itertools.product(
        masks, blur_kernels, blur_kernels, n_cluster_params
    ):
        print(mask, blur_kernel_1, blur_kernel_2, n_clusters)
        # umap_df = add_umap_components(df, masks=mask)
        h, w, _ = np.array(atlus.map_image()).shape
        orig_x, _, orig_y, _ = atlus.pixels_projected_extent(h, w)

        ref_img = rgb_to_binary(atlus, blur_kernel=blur_kernel_1)
        sub_img = spec_img_to_binary(umap_df, atlus, blur_kernel=blur_kernel_2)

        ref_img = np.array(ref_img)
        sub_img = np.array(sub_img)
        print(np.sum(sub_img == 255))
        print(sub_img.size)
        if 0.5 > (np.sum(sub_img == 255) / sub_img.size):
            print("Bigger")
            sub_img = 255 - sub_img
            print(sub_img)

        result = match_template(ref_img, sub_img, pad_input=False)
        print(np.max(result))
        ij = np.unravel_index(np.argmax(result), result.shape)
        x, y = ij[::-1]

        fig = plt.figure(figsize=(44, 10))
        ax1 = plt.subplot(1, 5, 1)
        ax2 = plt.subplot(1, 5, 2)
        ax3 = plt.subplot(1, 5, 3, sharex=ax2, sharey=ax2)
        ax4 = plt.subplot(1, 5, 4)
        ax5 = plt.subplot(1, 5, 5)

        ax1.imshow(ref_img, cmap=plt.cm.gray)
        ax1.set_axis_off()
        ax1.set_title("Image")

        ax2.imshow(sub_img, cmap=plt.cm.gray)
        ax2.set_axis_off()
        ax2.set_title("Spectra")
        # highlight matched region
        spec_h, spec_w = sub_img.shape
        pre = plt.Rectangle(
            (orig_x, orig_y), spec_w, spec_h, edgecolor="blue", facecolor="none"
        )
        post = plt.Rectangle((x, y), spec_w, spec_h, edgecolor="r", facecolor="none")
        ax1.add_patch(pre)
        ax1.add_patch(post)

        ax3.imshow(result)
        ax3.set_axis_off()
        ax3.set_title("`match_template`\nresult")
        # highlight matched region
        # ax3.autoscale(False)
        ax3.plot(x, y, "o", markeredgecolor="r", markerfacecolor="none", markersize=10)

        [xmin, xmax, ymin, ymax] = atlus._max_extent()

        diff_x = abs(xmax - xmin) * ((x - orig_x) / w)
        diff_y = abs(ymax - ymin) * ((y - orig_y) / h)

        atlus.plot_rgb_image(ax=ax4)
        # sns.scatterplot(
        #     data=umap_ctrl_df,
        #     x="pixel_x",
        #     y="pixel_y",
        #     hue="split",
        #     marker="s",
        #     linewidth=0,
        #     s=5,
        #     alpha=0.5,
        #     ax=ax4,
        #     palette="Blues"
        # )

        sns.scatterplot(
            data=umap_df.with_columns(
                pl.col("pixel_x").add(diff_x), pl.col("pixel_y").sub(diff_y)
            ),
            x="pixel_x",
            y="pixel_y",
            hue="split",
            marker="s",
            linewidth=0,
            s=5,
            alpha=0.5,
            ax=ax4,
            # palette="Reds"
        )

        # atlus.plot_rgb_image(ax=ax6)

        sns.scatterplot(
            data=umap_df,
            x="pixel_x",
            y="pixel_y",
            hue="split",
            marker="s",
            linewidth=0,
            s=5,
            alpha=0.5,
            ax=ax5,
            palette="Blues",
        )
        sns.scatterplot(
            data=umap_df.with_columns(
                pl.col("pixel_x").add(diff_x), pl.col("pixel_y").sub(diff_y)
            ),
            x="pixel_x",
            y="pixel_y",
            hue="split",
            marker="s",
            linewidth=0,
            s=5,
            alpha=0.5,
            ax=ax5,
            palette="Reds",
        )

        ax5.set_xlim(xmin, xmax)
        ax5.set_ylim(ymin, ymax)

        plt.show()


align_imgs_post(atlus=ctrl_atlus, umap_df=all_umap_df.filter(pl.col("cell_type") == "ctrl"))

In [ ]:
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq, ifft


def band_pass_filter(ss):
    wn = np.linspace(650, 4000, 3475)
    spectra = np.array(ss)

    d_wn = wn[1] - wn[0]  # Wavenumber spacing
    all_trans_fft = []
    for trans in spectra:
        all_trans_fft.append(fft(trans))
    freq = fftfreq(3475, d=d_wn)  # Frequency axis in the reciprocal wavenumber domain
    mask = (np.abs(freq) >= 0.0) & (np.abs(freq) <= 0.01)
    all_filtered_fft = [trans_fft * mask for trans_fft in all_trans_fft]
    all_filtered_trans = [
        np.real(ifft(filtered_fft)) for filtered_fft in all_filtered_fft
    ]
    return all_filtered_trans


def plot_cell_type(df: pl.DataFrame, cell_type: str, with_filter: bool = False):
    res = df.filter(pl.col("umap_1") >= 10)
    all_spec = res.filter(pl.col("cell_type") == cell_type).select(
        cs.contains("wavenumber")
    )
    all_spec_arr = all_spec.to_numpy()
    all_spec_arr = band_pass_filter(all_spec_arr)

    mean_spec = np.mean(all_spec_arr, axis=0)
    std_spec = np.std(all_spec_arr, axis=0)

    # std_spec = all_spec.std().to_numpy().ravel()
    x_axis = np.linspace(650, 4000, 3475)
    plt.plot(x_axis, mean_spec, label=cell_type)
    plt.fill_between(x_axis, mean_spec - std_spec, mean_spec + std_spec, alpha=0.5)
    print("test")


plot_cell_type(all_umap_df, "ctrl")
plot_cell_type(all_umap_df, "lps")
plot_cell_type(all_umap_df, "lta")
plt.legend()
plt.gca().invert_xaxis()

In [ ]:
plot_cell_type(all_umap_df, "ctrl")
plot_cell_type(all_umap_df, "lps")
plot_cell_type(all_umap_df, "lta")
plt.gca().invert_xaxis()
plt.xlim(1800, 700)

In [ ]:
lps_atlus.export_npz("LPS_sample_2_2hr_thp1_lps_Tr_32x_2.5stepsize_8resolution_2scans")

In [ ]:
import matplotlib.pyplot as plt
from pybaselines import Baseline

wavenumbers = np.linspace(650, 4000, 3475)
clip_mask = (900 <= wavenumbers) & (wavenumbers <= 3600)
wavenumbers = wavenumbers[clip_mask]

baseline_fitter = Baseline(x_data=wavenumbers)
# example_spectra = lps_atlus.spectra_dict[0]
example_spectra = 2 - np.log10(lps_atlus.spectra_dict[0])
example_spectra = example_spectra[clip_mask]
print(example_spectra.shape)
# plt.plot(
#     # wavenumbers
#     example_spectra,
# )
# res = baseline_fitter.arpls(example_spectra, lam=1e9)[0]
# res = baseline_fitter.imodpoly(example_spectra, poly_order=10, num_std=0.1)[0]
res = baseline_fitter.rubberband(example_spectra, segments=[900, 1900, 2250])[0]
# res = baseline_fitter.mor(example_spectra, half_window=150)[0]
# res = baseline_fitter.std_distribution(
#     example_spectra, half_window=15, smooth_half_window=10
# )[0]

# plt.plot(
#     wavenumbers,
#     res,
# )
plt.plot(
    wavenumbers,
    example_spectra - res,
)
plt.grid(True)
# plt.xticks(np.arange(0, 2800, 250))
plt.xlim(1500, 1800)

In [ ]:
from typing import Any, List
from matplotlib.figure import Figure
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq
from numpy.typing import ArrayLike


def band_pass_filter(ss):
    wn = np.linspace(650, 4000, 3475)
    spectra = np.array(ss)

    N = 3475
    d_wn = wn[1] - wn[0]  # Wavenumber spacing
    all_trans_fft = []
    for trans in spectra:
        all_trans_fft.append(fft(trans))
    freq = fftfreq(3475, d=d_wn)  # Frequency axis in the reciprocal wavenumber domain
    mask = (np.abs(freq) >= 0.0) & (np.abs(freq) <= 0.1)
    all_filtered_fft = [trans_fft * mask for trans_fft in all_trans_fft]
    all_filtered_trans = [
        np.real(ifft(filtered_fft)) for filtered_fft in all_filtered_fft
    ]
    return all_filtered_trans


def plot_spectra_fft(
    ss, low_cutoff=0.0, high_cutoff=0.01, plot_kwargs=None
) -> tuple[ArrayLike, tuple[Figure, Any]]:
    wn = np.linspace(650, 4000, 3475)
    spectra = np.array(ss)

    N = 3475
    d_wn = wn[1] - wn[0]  # Wavenumber spacing
    all_trans_fft = []
    for trans in spectra:
        all_trans_fft.append(fft(trans))
    freq = fftfreq(3475, d=d_wn)  # Frequency axis in the reciprocal wavenumber domain

    fig, ax = plt.subplots(
        2, 1, figsize=(18, 11), gridspec_kw=dict(height_ratios=[1, 4])
    )
    # plt.figure(figsize=(10, 5))
    for trans_fft in all_trans_fft:
        ax[0].plot(freq[: N // 2], np.abs(trans_fft[: N // 2]))
    ax[0].set_xlabel("Frequency (1/cm⁻¹)")
    ax[0].set_ylabel("Amplitude")
    ax[0].set_title("FFT Spectrum of FTIR Data")
    ax[0].grid(visible=True)

    # Create band-pass mask
    mask = (np.abs(freq) >= low_cutoff) & (np.abs(freq) <= high_cutoff)

    # Apply mask
    all_filtered_fft = [trans_fft * mask for trans_fft in all_trans_fft]

    all_filtered_trans = [
        np.real(ifft(filtered_fft)) for filtered_fft in all_filtered_fft
    ]

    default_plot_kwargs = dict(alpha=0.1, linewidth=0.1)
    if plot_kwargs is not None:
        default_plot_kwargs.update(plot_kwargs)

    # plt.figure(figsize=(12, 6))
    for trans in spectra:
        ax[1].plot(
            wn,
            trans,
            label="Original FTIR Spectra",
            c="blue",
            **default_plot_kwargs,
        )
    for filtered_trans in all_filtered_trans:
        ax[1].plot(
            wn,
            filtered_trans,
            label="Filtered FTIR Spectra",
            c="red",
            **default_plot_kwargs,
        )
    ax[1].set_xlabel("Wavenumber (cm⁻¹)")
    ax[1].set_ylabel("Transmission")
    ax[1].set_title("FTIR Spectra Before and After FFT Band-pass Filtering")
    # plt.legend()
    ax[1].invert_xaxis()  # FTIR spectra usually plotted with decreasing wavenumber
    ax[1].grid(visible=True)
    plt.tight_layout()
    # plt.show()
    return all_filtered_trans, (fig, ax)


plot_spectra_fft(
    list(lps_atlus.spectra_dict.values())[:10], plot_kwargs=dict(linewidth=1, alpha=0.8)
)

In [ ]:
all_filtered_trans, _ = plot_spectra_fft(
    ss=list(lps_atlus.spectra_dict.values()),
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots()

# Define the coordinates and colors for the squares
red = np.linspace(0, 1, 10)
blue = np.linspace(0, 1, 10)
squares = [
    dict(x=i, y=j, color=(r, b, 0.5))
    for (i, r) in enumerate(red)
    for (j, b) in enumerate(blue)
]
print(len(squares))

# squares = [
#     {'x': 0, 'y': 1, 'color': 'red'},
#     {'x': 0, 'y': 2, 'color': 'blue'},
#     {'x': 0, 'y': 3, 'color': 'green'},
#     {'x': 0, 'y': 4, 'color': 'yellow'}
# ]

# Iterate through the squares and plot them
for square in squares:
    rect = patches.Rectangle(
        (square["x"], square["y"]), 1, 1, facecolor=square["color"]
    )
    ax.add_patch(rect)

# Set the limits of the plot
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect("equal", adjustable="box")

# Show the plot
plt.show()

In [ ]:
wavenumbers = np.linspace(650, 4000, 3475)
protein_mask = (1100 < wavenumbers) & (wavenumbers < 1800)
protein_wn = wavenumbers[protein_mask]
spectra_arr_max_idx = np.argmax(np.array(all_filtered_trans)[:, protein_mask], axis=-1)
protein_wn[spectra_arr_max_idx]
# protein_wn
# spectra_arr = np.argmax()

In [ ]:
from itertools import product
from sklearn.preprocessing import MinMaxScaler
import umap


wavenumbers = np.linspace(650, 4000, 3475)
protein_mask = (1100 < wavenumbers) & (wavenumbers < 1800)
protein_finger = (1100 < wavenumbers) & (wavenumbers < 1800)
dna_mask = (800 < wavenumbers) & (wavenumbers < 1000)
all_mask = np.array([True] * 3475)
data_masks = [
    # ("all", all_mask),
    ("protein (1100 < wl < 1800)", protein_mask),
    ("protein-finger (800 < wl < 1800)", protein_finger),
    ("dna (800 < wl < 100)", dna_mask),
]

umap_neighbors = [
    # 5,
    15,
    30,
    60,
]
umap_metrics = ["euclidean", "correlation", "cosine"]
umap_densmap = [False, True]
umap_min_dist = [0.1, 0.5, 0.9]
for (mask_name, mask), n_neighbors, metric, densmap, min_dist in product(
    data_masks, umap_neighbors, umap_metrics, umap_densmap, umap_min_dist
):
    wn_scaler = MinMaxScaler()
    spectra_arr = np.array(all_filtered_trans)[:, mask]
    spectra_arr = (spectra_arr - spectra_arr.min()) / (
        spectra_arr.max() - spectra_arr.min()
    )
    max_spectra_arr = np.max(spectra_arr, axis=-1)
    protein_wn = wn_scaler.fit_transform(wavenumbers[mask].reshape(-1, 1))
    spectra_arr_max_idx = protein_wn[np.argmax(spectra_arr, axis=-1), :]

    default_color_code = 0.5
    max_spectra_colors = [
        (x, default_color_code, default_color_code) for x in max_spectra_arr
    ]
    spectra_arr_max_idx_colors = [
        (default_color_code, x, default_color_code) for x in spectra_arr_max_idx.ravel()
    ]
    combined_colors = [
        (a, b, default_color_code)
        for a, b in zip(max_spectra_arr, spectra_arr_max_idx.ravel())
    ]

    components = umap.UMAP(
        n_neighbors=n_neighbors,
        metric=metric,
        densmap=densmap,
        min_dist=min_dist,
    ).fit_transform(spectra_arr)
    plt.scatter(components[:, 0], components[:, 1], color=combined_colors, s=2)
    plt.title(
        f"Data Mask={mask_name},\nUMAP(n_neighbors={n_neighbors}, metric={metric}, densmap={densmap}, min_dist={min_dist})"
    )
    plt.show()

In [ ]:
# Synthetic example data (replace this with your actual data)
wn = np.linspace(4000, 400, 3600)  # wavenumber from 4000 to 400 cm⁻¹
trans = np.exp(-((wn - 1700) ** 2) / (2 * 50**2)) + 0.05 * np.random.randn(
    len(wn)
)  # Example peak at ~1700 cm⁻¹

N = len(trans)
d_wn = wn[1] - wn[0]  # Wavenumber spacing
trans_fft = fft(trans)
freq = fftfreq(N, d=d_wn)  # Frequency axis in the reciprocal wavenumber domain

plt.figure(figsize=(10, 5))
plt.plot(freq[: N // 2], np.abs(trans_fft[: N // 2]))
plt.xlabel("Frequency (1/cm⁻¹)")
plt.ylabel("Amplitude")
plt.title("FFT Spectrum of FTIR Data")
plt.grid()
plt.show()

low_cutoff = 0.001  # Lower frequency cutoff (1/cm⁻¹)
high_cutoff = 0.01  # Higher frequency cutoff (1/cm⁻¹)

# Create band-pass mask
mask = (np.abs(freq) >= low_cutoff) & (np.abs(freq) <= high_cutoff)

# Apply mask
filtered_fft = trans_fft * mask

filtered_trans = np.real(ifft(filtered_fft))

plt.figure(figsize=(12, 6))
plt.plot(wn, trans, label="Original FTIR Spectra", alpha=0.6)
plt.plot(wn, filtered_trans, label="Filtered FTIR Spectra", linewidth=2)
plt.xlabel("Wavenumber (cm⁻¹)")
plt.ylabel("Transmission")
plt.title("FTIR Spectra Before and After FFT Band-pass Filtering")
plt.legend()
plt.gca().invert_xaxis()  # FTIR spectra usually plotted with decreasing wavenumber
plt.grid()
plt.show()